# Lấy mẫu URL pattern để đánh giá độ sạch của Crawl4AI — chạy local

Giữ nguyên cấu hình browser, fast/protected/slow, pruning `dynamic=0.6`,
HTML → Markdown, FIT → RAW fallback và phân loại lỗi của notebook gốc.
Không có Kaggle, Google Drive, chia worker hoặc lịch sử crawl toàn corpus.

**Cách chạy:** đặt file JSON của bạn cạnh notebook, sửa `PATTERNS_FILE` ở mục 2,
rồi chạy lần lượt các cell. Máy cần Internet, Python 3.10+ và Chromium của Crawl4AI.
Nếu Crawl4AI đã được cài và chạy được, bỏ qua cell cài đặt.

URL được lấy từ **corpus `AIGuruTinix/ViBioMIR` như notebook gốc**.
Notebook chỉ quét danh sách URL một lần để chọn mẫu; không crawl toàn corpus.
Mỗi pattern giữ tối đa 30 URL thật làm ứng viên, crawl đến khi có 10 kết quả
thành công hoặc hết ứng viên. Với đúng 242 pattern và đủ kết quả thành công,
`dataset.jsonl` có 2.420 dòng. Nếu thiếu, báo số thiếu, không lặp bài hay tạo dòng giả.

`success=True` giữ đúng nghĩa của notebook gốc: crawl được Markdown và không
phát hiện trang chặn. **Đây chưa phải nhãn “nội dung sạch” hoặc “đúng bài viết”.**
LLM sẽ đánh giá điều đó sau. Log tất cả lần thử giúp thấy cả những pattern bị lỗi.

## 1. Cài đặt local (chỉ cần lần đầu)

In [ ]:
%pip install -q -U crawl4ai datasets tqdm pandas
!crawl4ai-setup


## 2. Cấu hình và đọc file pattern

File phải là JSON hợp lệ (dấu nháy kép, không có dấu phẩy cuối):

```json
[
  {"domain": "cnkang.com", "urls": ["/myjk/*/*.html", "/dise/74/*.html", "/dise/*.html"]}
]
```

`domain` khớp hostname, bỏ `www.`; không tự gộp subdomain khác.
`*` khớp chuỗi bất kỳ, **kể cả `/`**. Vì vậy `/dise/*.html` có thể chồng lấn
với `/dise/74/*.html`. Một URL được phép xuất hiện ở hai pattern để mỗi pattern
có bộ mẫu riêng, nhưng trong cùng một pattern thì URL không lặp.
Pattern có `?` được so với cả path + query; nếu không thì bỏ query khi khớp.
Chọn mẫu bằng hash + seed để có kết quả ổn định, tránh chỉ lấy 10 URL đầu corpus.

In [ ]:
import json
import re
import heapq
import hashlib
import csv
from pathlib import Path
from collections import defaultdict, deque
from urllib.parse import urlsplit, urlparse
from datetime import datetime, timezone
from importlib.metadata import version
from tqdm.auto import tqdm

PATTERNS_FILE = Path("/Volumes/security/lucraft@lucraftmeds/lucraftmeds@notebooks/data-crawl/domains-analyzation/domain_profiling_output/domain_urls.json")   # Sửa thành tên file JSON của bạn.
OUTPUT_DIR = Path("pattern_cleanliness")
SAMPLES_PER_PATTERN = 10
CANDIDATES_PER_PATTERN = 30                # Bao gồm 10 mẫu chính + URL dự phòng.
SEED = 42
MAX_CONCURRENCY = 3                        # Số trang chạy đồng thời trên máy local.
MEMORY_THRESHOLD_PERCENT = 80.0
DATASET_ID = "AIGuruTinix/ViBioMIR"
DATASET_CONFIG = "corpus"

PROTECTED_DOMAINS = {
    'special.vietnamplus.vn',
    'cnkang.com',
    '120ask.com',
    'familydoctor.com.cn',
    'ask.39.net',
    'vov2.vov.vn',
    'a-hospital.com',
    'suckhoecongdongonline.vn',
    'zhongyibaodian.net',
    'suckhoecongdongonline.vn',
    'zydcd.com',
    'thanhnien.vn',
    'wujue.com',
    'youlai.cn',
    'vov.vn',
    'zysjonline.com',
    'pmc-ecm-healthblog.beta.pharmacity.io',
    'nhathuoclongchau.com.vn',
    'giadinhonline.vn',
    'ask.familydoctor.com.cn',
}

SLOW_DOMAINS = {
    'bachmai.gov.vn',
    'wujue.com',
    'test.pmphai.com',
    'qihuangzhishu.com',
    'woman.39.net',
    'tnb.39.net',
    'wei.39.net',
    'shen.39.net',
    'baidu.com',
    'sj.39.net',
    'v.familydoctor.com.cn',
    'yanglao.familydoctor.com.cn',
    'ypk.familydoctor.com.cn',
    'baothanhhoa.vn',
    
}

if SAMPLES_PER_PATTERN <= 0 or CANDIDATES_PER_PATTERN < SAMPLES_PER_PATTERN:
    raise ValueError("Cần 0 < SAMPLES_PER_PATTERN <= CANDIDATES_PER_PATTERN.")
if MAX_CONCURRENCY <= 0:
    raise ValueError("MAX_CONCURRENCY phải lớn hơn 0.")

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def load_patterns(path):
    groups = json.loads(path.read_text(encoding="utf-8-sig"))
    if not isinstance(groups, list) or not groups:
        raise ValueError("File pattern phải là một list JSON không rỗng.")
    patterns, seen = [], set()
    for group in groups:
        if not isinstance(group, dict):
            raise ValueError("Mỗi phần tử phải có domain và urls.")
        domain = group.get("domain", "")
        paths = group.get("urls", [])
        if not isinstance(domain, str) or not isinstance(paths, list) or not paths:
            raise ValueError("domain phải là chuỗi, urls phải là list không rỗng.")
        domain = domain.strip().lower().removeprefix("www.")
        if not domain or any(char in domain for char in "/:*?# "):
            raise ValueError(f"domain chỉ chứa hostname, không chứa scheme/path: {domain!r}")
        for pattern in paths:
            if not isinstance(pattern, str) or not pattern.startswith("/"):
                raise ValueError(f"Pattern phải là path bắt đầu bằng /: {pattern!r}")
            if (domain, pattern) in seen:
                raise ValueError(f"Pattern bị lặp: {domain}{pattern}")
            seen.add((domain, pattern))
            patterns.append({"pattern_id": f"p{len(patterns) + 1:03d}",
                             "pattern_domain": domain, "url_pattern": pattern})
    return patterns

patterns = load_patterns(PATTERNS_FILE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_ROWS = len(patterns) * SAMPLES_PER_PATTERN
print(f"{len(patterns)} pattern × {SAMPLES_PER_PATTERN} mẫu = mục tiêu {TARGET_ROWS:,} dòng.")
if len(patterns) != 242:
    print("Lưu ý: file thực tế không có 242 pattern; mục tiêu tính theo số pattern đọc được.")


## 3. Chọn URL thật cho từng pattern

Lần đầu quét hết cột URL của corpus bằng streaming và chỉ giữ tối đa
`CANDIDATES_PER_PATTERN` URL khác nhau cho mỗi pattern trong RAM. Không dừng ở
10 URL đầu tiên. Hash nhỏ nhất được giữ lại; seed cố định cho kết quả lặp lại.
`matched_source_rows` đếm dòng corpus khớp, có thể bao gồm URL trùng trong nguồn.

Lưu `selected_urls.json` để những lần sau không phải quét lại. Nếu đổi pattern,
seed hoặc số ứng viên, hãy chọn `OUTPUT_DIR` mới để giữ kết quả cũ.
Quét vài triệu dòng có thể mất thời gian và vẫn dùng băng thông của dataset.

In [ ]:
from datasets import load_dataset

def select_candidates(rows, patterns, capacity, seed):
    by_domain = defaultdict(list)
    for index, spec in enumerate(patterns):
        regex = re.compile(re.escape(spec["url_pattern"]).replace(r"\*", ".*"))
        by_domain[spec["pattern_domain"]].append((index, regex, "?" in spec["url_pattern"]))
    heaps = [[] for _ in patterns]
    selected = [set() for _ in patterns]
    counts = [0] * len(patterns)

    for row in tqdm(rows, desc="Quét URL để chọn mẫu", unit="URL"):
        url = row.get("url")
        if not isinstance(url, str) or not url.strip():
            continue
        url = url.strip()
        try:
            parsed = urlsplit(url)
            domain = (parsed.hostname or "").lower().removeprefix("www.")
        except ValueError:
            continue
        if parsed.scheme not in {"http", "https"} or domain not in by_domain:
            continue
        score = int.from_bytes(hashlib.sha256(f"{seed}:{url}".encode()).digest()[:8], "big")
        entry = (-score, url, str(row.get("id", "")))
        for index, regex, include_query in by_domain[domain]:
            path = parsed.path or "/"
            if include_query and parsed.query:
                path += "?" + parsed.query
            if not regex.fullmatch(path):
                continue
            counts[index] += 1
            if url in selected[index]:
                continue
            heap = heaps[index]
            if len(heap) < capacity:
                heapq.heappush(heap, entry)
                selected[index].add(url)
            elif entry > heap[0]:
                removed = heapq.heapreplace(heap, entry)
                selected[index].remove(removed[1])
                selected[index].add(url)

    return [{**spec, "matched_source_rows": counts[index],
             "candidates": [{"url": url, "id": doc_id}
                            for _, url, doc_id in sorted(heaps[index], reverse=True)]}
            for index, spec in enumerate(patterns)]

selection_settings = {"patterns": patterns, "dataset": DATASET_ID,
                      "config": DATASET_CONFIG, "split": "train",
                      "seed": SEED, "capacity": CANDIDATES_PER_PATTERN}
selection_path = OUTPUT_DIR / "selected_urls.json"
if selection_path.exists():
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    if selection["settings"] != selection_settings:
        raise ValueError("Cấu hình chọn mẫu đã đổi. Chọn OUTPUT_DIR mới rồi chạy lại.")
    pattern_samples = selection["patterns"]
    print("Dùng lại danh sách URL đã chọn.")
else:
    source = load_dataset(DATASET_ID, DATASET_CONFIG, split="train", streaming=True)
    pattern_samples = select_candidates(source, patterns, CANDIDATES_PER_PATTERN, SEED)
    selection = {"settings": selection_settings, "patterns": pattern_samples}
    write_json(selection_path, selection)

short_candidates = [p for p in pattern_samples if len(p["candidates"]) < SAMPLES_PER_PATTERN]
print(f"Đã chọn {sum(len(p['candidates']) for p in pattern_samples):,} URL ứng viên theo pattern.")
for spec in short_candidates:
    print(f"THIẾU URL: {spec['pattern_id']} {spec['pattern_domain']}{spec['url_pattern']} "
          f"chỉ có {len(spec['candidates'])}/{SAMPLES_PER_PATTERN} ứng viên.")


## 4. Cấu hình crawl và chuyển Markdown — giữ nguyên notebook gốc

Thứ tự ưu tiên vẫn là **protected → slow → fast**. Domain nằm trong cả hai
danh sách dùng protected như trước. Không thêm bộ lọc làm sạch mới.
Ba kết quả dùng cho đánh giá: `markdown` là đầu ra hiện tại;
`raw_markdown` và `fit_markdown` là hai bản trước/sau pruning để đối chiếu.

In [ ]:
def first_metadata_value(metadata: dict, *keys):
    for key in keys:
        value = metadata.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


In [ ]:
from urllib.parse import urlparse, urlsplit

from crawl4ai import (
    AsyncWebCrawler,
    BrowserConfig,
    CacheMode,
    CrawlerRunConfig,
    MemoryAdaptiveDispatcher,
    RateLimiter,
)
from crawl4ai.content_filter_strategy import PruningContentFilterLXML
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

browser_config = BrowserConfig(
    browser_type='chromium',
    headless=True,
    enable_stealth=True,
    user_agent_mode='random',
)

prune_filter = PruningContentFilterLXML(
    threshold=0.6,
    threshold_type='dynamic',
    min_word_threshold=25,
)

markdown_generator = DefaultMarkdownGenerator(
    content_source='cleaned_html',
    content_filter=prune_filter,
    options={'ignore_links': True, "ignore_images": True},
)

# Fast default for most URLs. Streaming lets us checkpoint each completed result.
fast_config = CrawlerRunConfig(
    stream=True,
    cache_mode=CacheMode.BYPASS,
    wait_until='domcontentloaded',
    page_timeout=45_000,
    delay_before_return_html=0.2,
    markdown_generator=markdown_generator,
    excluded_tags=["nav", "footer", "aside"],
)

# Expensive behavior is restricted to known protected domains.
protected_config = CrawlerRunConfig(
    url_matcher=lambda url: urlparse(url).netloc.lower().removeprefix('www.') in PROTECTED_DOMAINS,
    stream=True,
    cache_mode=CacheMode.BYPASS,
    magic=True,
    simulate_user=True,
    override_navigator=True,
    page_timeout=90_000,
    delay_before_return_html=3.0,
    max_retries=2,
    markdown_generator=markdown_generator,
    excluded_tags=["nav", "footer", "aside"],
)

slow_config = CrawlerRunConfig(
    url_matcher=lambda url: (
        urlparse(url).netloc
        .lower()
        .removeprefix("www.") in SLOW_DOMAINS
    ),

    stream=True,
    cache_mode=CacheMode.BYPASS,

    # Chỉ chờ server bắt đầu trả document,
    # không bắt buộc DOMContentLoaded phải hoàn tất.
    wait_until="commit",

    page_timeout=90_000,

    # Sau commit, đợi nội dung body thực sự xuất hiện.
    wait_for="""
        js:() => {
            return document.body
                && document.body.innerText
                && document.body.innerText.length > 500;
        }
    """,
    wait_for_timeout=30_000,
    delay_before_return_html=1.0,
    excluded_tags=["nav", "footer", "aside"],
    markdown_generator=markdown_generator,
)

crawl_configs = [
    protected_config,  # Anti-bot domains
    slow_config,       # Slow-loading domains
    fast_config,       # Default
]

def make_dispatcher():
    return MemoryAdaptiveDispatcher(
        memory_threshold_percent=MEMORY_THRESHOLD_PERCENT,
        max_session_permit=MAX_CONCURRENCY,
        rate_limiter=RateLimiter(
            base_delay=(0.2, 0.8),
            max_delay=20.0,
            max_retries=2,
        ),
    )

def markdown_text(markdown_result):
    if markdown_result is None:
        return ""

    if isinstance(markdown_result, str):
        return markdown_result

    raw = markdown_result.raw_markdown or ""
    fit = markdown_result.fit_markdown or ""

    # FIT chỉ chứa "word" lặp lại → dùng RAW.
    if fit.strip() and set(fit.split()) != {"word"}:
        return fit

    return raw

def url_identity(url: str) -> tuple[str, str, str]:
    parsed = urlsplit(url or '')
    host = parsed.netloc.lower().removeprefix('www.')
    path = parsed.path.rstrip('/') or '/'
    return host, path, parsed.query

def blocked_page_reason(title, markdown, status_code):
    title_text = (title or '').strip().lower()
    content = (markdown or '').strip().lower()
    if status_code in {403, 429, 503}:
        return f'http_{status_code}'
    blocked_titles = {'checking your browser...', 'just a moment...', 'access denied'}
    if title_text in blocked_titles:
        return 'anti_bot_challenge'
    markers = ('checking your browser', 'verify you are human', 'cf-chl-', 'captcha')
    if len(content) < 2_000 and any(marker in content for marker in markers):
        return 'anti_bot_challenge'
    return None

def crawl_result_to_record(result, requested_url: str) -> dict:
    status_code = getattr(result, 'status_code', None)
    redirected_url = getattr(result, 'redirected_url', None)
    final_url = redirected_url or getattr(result, 'url', None) or requested_url
    final_status_code = getattr(result, 'redirected_status_code', None) or status_code
    base = {
        'url': requested_url,
        'final_url': final_url,
        'domain': urlparse(final_url).netloc.lower(),
        'status_code': status_code,
        'final_status_code': final_status_code,
        'redirected': final_url != requested_url,
    }
    if not result.success:
        return {
            **base,
            'success': False,
            'error_type': 'crawl_error',
            'error': getattr(result, 'error_message', None) or 'Unknown crawl error',
        }

    metadata = getattr(result, 'metadata', None) or {}
    title = first_metadata_value(metadata, 'title', 'og:title', 'twitter:title')
    description = first_metadata_value(
        metadata, 'description', 'og:description', 'twitter:description'
    )
    keywords = first_metadata_value(metadata, 'keywords')
    markdown = markdown_text(getattr(result, 'markdown', None))
    blocked_reason = blocked_page_reason(title, markdown, final_status_code)

    if blocked_reason:
        return {
            **base,
            'title': title,
            'description': description,
            'keywords': keywords,
            'markdown': markdown,
            'success': False,
            'retryable': True,
            'error_type': blocked_reason,
            'error': 'Received an anti-bot or rate-limit page instead of article content.',
        }
    if not markdown.strip():
        return {
            **base,
            'title': title,
            'success': False,
            'retryable': True,
            'error_type': 'empty_content',
            'error': 'Crawl succeeded but produced no Markdown.',
        }
    return {
        **base,
        'title': title,
        'description': description,
        'keywords': keywords,
        'markdown': markdown,
        'success': True,
    }


## 5. Crawl tối đa 10 mẫu thành công mỗi pattern và lưu checkpoint local

Browser đóng sau mỗi batch, như notebook gốc. Mỗi batch chỉ lấy số URL còn
thiếu nên không vượt quá 10 mẫu thành công/pattern. Lỗi được giữ trong log và
dùng URL dự phòng tiếp theo; không thử vô hạn.

Checkpoint theo pattern được ghi sau mỗi batch. Rerun cell tiếp tục từ những
URL chưa thử; không nối thêm dòng trùng. Nếu tắt máy trong một batch, chỉ batch
chưa được ghi có thể phải chạy lại. Khi đổi cấu hình crawl hoặc cập nhật
Crawl4AI, dùng `OUTPUT_DIR` mới để không trộn các thí nghiệm.

In [ ]:
def markdown_diagnostics(result):
    value = getattr(result, "markdown", None)
    raw = value if isinstance(value, str) else (getattr(value, "raw_markdown", None) or "")
    fit = "" if isinstance(value, str) else (getattr(value, "fit_markdown", None) or "")
    use_fit = bool(fit.strip()) and set(fit.split()) != {"word"}
    source = "fit" if use_fit else ("raw" if raw else "empty")
    return {"raw_markdown": raw, "fit_markdown": fit, "markdown_source": source}

from collections import defaultdict, deque
from tqdm.auto import tqdm
import asyncio

def pending_lookup(batch):
    lookup = defaultdict(deque)
    for item in batch:
        lookup[url_identity(item['url'])].append(item)
    return lookup

def remaining_items(lookup):
    for queue in lookup.values():
        yield from queue

async def crawl_batch(crawler, dispatcher, batch, progress):
    records = []
    lookup = pending_lookup(batch)
    successes = errors = unmatched_results = 0

    try:
        result_stream = await crawler.arun_many(
            urls=[item['url'] for item in batch],
            config=crawl_configs,
            dispatcher=dispatcher,
        )

        async for crawl_result in result_stream:
            result_url = getattr(crawl_result, 'url', None) or ''
            queue = lookup.get(url_identity(result_url))
            if not queue:
                # Do not guess a doc_id when a returned URL cannot be mapped safely.
                unmatched_results += 1
                continue

            item = queue[0]
            doc_id = str(item['id'])
            record = crawl_result_to_record(crawl_result, item['url'])
            record['doc_id'] = doc_id
            record.update(markdown_diagnostics(crawl_result))
            queue.popleft()

            if record['success']:
                records.append(record)
                successes += 1
            else:
                records.append(record)
                errors += 1

            progress.update(1)

    except Exception as exc:
        batch_error = f'{type(exc).__name__}: {exc}'
        for item in remaining_items(lookup):
            doc_id = str(item['id'])
            record = {
                'doc_id': doc_id,
                'url': item['url'],
                'success': False,
                'retryable': True,
                'error_type': 'batch_error',
                'error': batch_error,
            }
            records.append(record)
            errors += 1
            progress.update(1)
        return records, successes, errors, unmatched_results

    # A missing/unmappable result is retried later instead of being linked to the wrong doc_id.
    for item in remaining_items(lookup):
        doc_id = str(item['id'])
        record = {
            'doc_id': doc_id,
            'url': item['url'],
            'success': False,
            'retryable': True,
            'error_type': 'unmapped_result',
            'error': 'arun_many returned no safely mappable result for this URL.',
        }
        records.append(record)
        errors += 1
        progress.update(1)

    return records, successes, errors, unmatched_results


CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Ghi cấu hình chính để không resume nhầm thư mục thí nghiệm.
config_fields = ("wait_until", "page_timeout", "delay_before_return_html", "magic",
                 "simulate_user", "override_navigator", "wait_for", "wait_for_timeout",
                 "max_retries", "excluded_tags")
run_settings = {"selection_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
                "target_per_pattern": SAMPLES_PER_PATTERN,
                "crawl_configs": [{name: getattr(config, name, None) for name in config_fields}
                                  for config in crawl_configs],
                "pruning": {name: getattr(prune_filter, name, None)
                            for name in ("threshold", "threshold_type", "min_word_threshold")},
                "protected_domains": sorted(PROTECTED_DOMAINS),
                "slow_domains": sorted(SLOW_DOMAINS),
                "max_concurrency": MAX_CONCURRENCY,
                "memory_threshold_percent": MEMORY_THRESHOLD_PERCENT,
                "crawl4ai_version": version("crawl4ai")}
manifest_path = OUTPUT_DIR / "run_settings.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != run_settings:
        raise ValueError("Cấu hình crawl đã đổi. Dùng OUTPUT_DIR mới để chạy thí nghiệm mới.")
else:
    write_json(manifest_path, run_settings)

def checkpoint_path(spec):
    return CHECKPOINT_DIR / f"{spec['pattern_id']}.json"

def read_attempts(spec):
    path = checkpoint_path(spec)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else []

def export_outputs():
    # Tạo lại output từ checkpoint, luôn giữ phần đã crawl xong.
    summary = []
    dataset_path = OUTPUT_DIR / "dataset.jsonl"
    attempts_path = OUTPUT_DIR / "crawl_attempts.jsonl"
    dataset_temp = dataset_path.with_suffix(".jsonl.tmp")
    attempts_temp = attempts_path.with_suffix(".jsonl.tmp")
    with dataset_temp.open("w", encoding="utf-8") as dataset_file, \
         attempts_temp.open("w", encoding="utf-8") as attempts_file:
        for spec in pattern_samples:
            attempts = read_attempts(spec)
            successes = [row for row in attempts if row["success"]]
            for row in attempts:
                attempts_file.write(json.dumps(row, ensure_ascii=False) + "\n")
            for index, row in enumerate(successes[:SAMPLES_PER_PATTERN], 1):
                sample = {**row, "sample_id": f"{spec['pattern_id']}_{index:02d}",
                          "sample_index": index}
                dataset_file.write(json.dumps(sample, ensure_ascii=False) + "\n")
            summary.append({"pattern_id": spec["pattern_id"],
                            "pattern_domain": spec["pattern_domain"],
                            "url_pattern": spec["url_pattern"],
                            "matched_source_rows": spec["matched_source_rows"],
                            "candidate_urls": len(spec["candidates"]),
                            "attempted": len(attempts), "successes": len(successes),
                            "errors": len(attempts) - len(successes),
                            "missing_samples": max(0, SAMPLES_PER_PATTERN - len(successes)),
                            "untried_candidates": len(spec["candidates"]) - len(attempts)})
    dataset_temp.replace(dataset_path)
    attempts_temp.replace(attempts_path)
    with (OUTPUT_DIR / "pattern_summary.csv").open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary[0]))
        writer.writeheader()
        writer.writerows(summary)
    return summary

async def run_sampling():
    progress = tqdm(desc="URL đã thử", unit="URL")
    try:
        for spec in pattern_samples:
            attempts = read_attempts(spec)
            tried = {row["url"] for row in attempts}
            remaining = [item for item in spec["candidates"] if item["url"] not in tried]
            successes = sum(row["success"] for row in attempts)
            while successes < SAMPLES_PER_PATTERN and remaining:
                needed = SAMPLES_PER_PATTERN - successes
                batch, remaining = remaining[:needed], remaining[needed:]
                async with AsyncWebCrawler(config=browser_config) as crawler:
                    records, _, _, _ = await crawl_batch(crawler, make_dispatcher(), batch, progress)
                for row in records:
                    row.update({"pattern_id": spec["pattern_id"],
                                "pattern_domain": spec["pattern_domain"],
                                "url_pattern": spec["url_pattern"],
                                "crawled_at": datetime.now(timezone.utc).isoformat()})
                    for key in ("title", "description", "keywords", "status_code", "final_status_code"):
                        row.setdefault(key, None)
                    for key in ("markdown", "raw_markdown", "fit_markdown"):
                        row.setdefault(key, "")
                    row.setdefault("markdown_source", "empty")
                    row.setdefault("final_url", row["url"])
                    row.setdefault("domain", urlparse(row["final_url"]).netloc.lower())
                    row.setdefault("redirected", row["final_url"] != row["url"])
                    row.setdefault("error_type", None)
                    row.setdefault("error", None)
                    row["markdown_chars"] = len(row["markdown"])
                    row["raw_markdown_chars"] = len(row["raw_markdown"])
                    row["fit_markdown_chars"] = len(row["fit_markdown"])
                attempts.extend(records)
                successes = sum(row["success"] for row in attempts)
                write_json(checkpoint_path(spec), attempts)
                progress.set_postfix(pattern=spec["pattern_id"], ok=successes)
            export_outputs()
            tqdm.write(f"{spec['pattern_id']} {spec['pattern_domain']}{spec['url_pattern']}: "
                       f"{successes}/{SAMPLES_PER_PATTERN} mẫu, {len(attempts)} lần thử")
    finally:
        progress.close()
        export_outputs()

await run_sampling()
summary_rows = export_outputs()
actual = sum(row["successes"] for row in summary_rows)
missing = sum(row["missing_samples"] for row in summary_rows)
print(f"Dataset: {actual:,}/{TARGET_ROWS:,} dòng thành công. Còn thiếu: {missing:,}.")
print("Output:", OUTPUT_DIR.resolve())


## 6. Kiểm tra dataset và các pattern thiếu mẫu

| File | Nội dung |
|---|---|
| `dataset.jsonl` | Tối đa 10 dòng thành công/pattern; Markdown đầy đủ, không cắt ngắn |
| `crawl_attempts.jsonl` | Tất cả lần thử, gồm thành công và lỗi; dùng kiểm tra tỷ lệ crawl lỗi |
| `pattern_summary.csv` | Số URL nguồn khớp, ứng viên, lần thử, thành công, lỗi và số mẫu thiếu |
| `selected_urls.json` | URL thật đã chọn cùng `id` nguồn, có thể dùng lại |
| `run_settings.json` | Cấu hình và phiên bản Crawl4AI của lần chạy |

Mỗi dòng dataset có `pattern_id`, `pattern_domain`, `url_pattern`, `sample_id`,
`doc_id`, URL gốc/cuối, HTTP status, title/description/keywords, `markdown`,
`raw_markdown`, `fit_markdown`, `markdown_source`, độ dài và thời điểm crawl.

Cho LLM đọc **`markdown`** để chấm đầu ra hiện tại; dùng RAW/FIT để điều tra khi
mất nội dung hoặc còn nhiều menu. Không chỉ đếm số dòng: kiểm tra tất cả pattern
có đủ mẫu và đọc `crawl_attempts.jsonl` để tránh bỏ qua các pattern khó crawl.
Các mẫu đã thành công là tập thiên về trang crawl được, không đại diện cho tỷ lệ
thành công của toàn bộ website. 10 mẫu/template chỉ là kiểm tra ban đầu; nếu bạn
điều chỉnh rule bằng những mẫu này, hãy kiểm tra lại với một bộ mẫu khác.

In [ ]:
import pandas as pd
from itertools import islice

summary_df = pd.read_csv(OUTPUT_DIR / "pattern_summary.csv")
display(summary_df)
shortages = summary_df[summary_df["missing_samples"] > 0]
if not shortages.empty:
    print("Các pattern chưa đủ mẫu:")
    display(shortages)

with (OUTPUT_DIR / "dataset.jsonl").open(encoding="utf-8") as f:
    sample_rows = [json.loads(line) for line in islice(f, 3)]
display(pd.DataFrame(sample_rows).drop(columns=["markdown", "raw_markdown", "fit_markdown"], errors="ignore"))
if sample_rows:
    print("Markdown mẫu đầu (chỉ preview 2.000 ký tự; file giữ nguyên toàn bộ):")
    print(sample_rows[0]["markdown"][:2000])


Tài liệu API đã đối chiếu:
[Crawl4AI arun_many](https://docs.crawl4ai.com/api/arun_many/),
[Crawl4AI cấu hình và thứ tự URL matcher](https://docs.crawl4ai.com/api/parameters/),
[Hugging Face streaming](https://huggingface.co/docs/datasets/en/stream).